In [40]:
import networkx as nx
import pandas as pd
import itertools
from joblib import Parallel, delayed
import numpy as np
from tqdm import tqdm
from node2vec import Node2Vec
import time
import community as community_louvain  

In [41]:
G = nx.read_gml("../data/GraphMissingEdges.gml")
df = pd.read_csv('../data/edgesToEvaluate.csv')
df['u'] = df['venue1']
df['v'] = df['venue2']
df = df.drop(columns=['linkID', 'venue1', 'venue2'], axis=1)

In [42]:
df.head()

,u,v
0,mJ_ucQ2_3hfTsmCcKb-hgw,qXGKYRwCR9SLgLl0g_9o5g
1,y19xFolCozaRA-gGmHwkQA,F6c3D1o9Z4Tl6cDorb3WgA
2,R1GwW4C1gh2Nmue9K0WYVA,Ul6JwluSTm12PVDIqnNaTg
3,zzBa0pQjM1gov00bXjYYXg,3D6Uck9QSdxZKFstf5DGlg
4,U2d-meX4sVq0kiqcrpHt1w,vuDL_d3GYAtbvX9EJQqVog


In [43]:

degreeCentrality = nx.degree_centrality(G)
betCentrality = nx.betweenness_centrality(G)
cloCentrality = nx.closeness_centrality(G)
eigCentrality = nx.eigenvector_centrality(G)

# NOVAS CENTRALIDADES para melhorar desempenho
print("- PageRank centrality...")
pageRankCentrality = nx.pagerank(G, alpha=0.85)

print("- Katz centrality...")
try:
    katzCentrality = nx.katz_centrality(G, alpha=0.1, beta=1.0)
except:
    # Fallback se não convergir
    katzCentrality = nx.katz_centrality_numpy(G, alpha=0.1, beta=1.0)

print("- Load centrality (amostra)...")
# Load centrality é custoso, então calculamos para uma amostra representativa
sample_nodes = list(G.nodes())[:1000] if len(G.nodes()) > 1000 else list(G.nodes())
loadCentrality_sample = nx.load_centrality(G.subgraph(sample_nodes))
# Extrapolar para todos os nós com valor médio para os não calculados
avg_load = np.mean(list(loadCentrality_sample.values()))
loadCentrality = {node: loadCentrality_sample.get(node, avg_load) for node in G.nodes()}

print("Aplicando centralidades aos pares...")

# Centralidades básicas
df['u_deg'] = df['u'].map(degreeCentrality)
df['v_deg'] = df['v'].map(degreeCentrality)
df['u_bet'] = df['u'].map(betCentrality)
df['v_bet'] = df['v'].map(betCentrality)
df['u_clo'] = df['u'].map(cloCentrality)
df['v_clo'] = df['v'].map(cloCentrality)
df['u_eig'] = df['u'].map(eigCentrality)
df['v_eig'] = df['v'].map(eigCentrality)

# NOVAS centralidades
df['u_pagerank'] = df['u'].map(pageRankCentrality)
df['v_pagerank'] = df['v'].map(pageRankCentrality)
df['u_katz'] = df['u'].map(katzCentrality)
df['v_katz'] = df['v'].map(katzCentrality)
df['u_load'] = df['u'].map(loadCentrality)
df['v_load'] = df['v'].map(loadCentrality)

# Features derivadas das centralidades (diferenças e ratios)
print("Calculando features derivadas...")
df['deg_diff'] = np.abs(df['u_deg'] - df['v_deg'])
df['bet_diff'] = np.abs(df['u_bet'] - df['v_bet'])
df['pagerank_diff'] = np.abs(df['u_pagerank'] - df['v_pagerank'])
df['katz_diff'] = np.abs(df['u_katz'] - df['v_katz'])

# Ratios (evitando divisão por zero)
df['deg_ratio'] = np.minimum(df['u_deg'], df['v_deg']) / (np.maximum(df['u_deg'], df['v_deg']) + 1e-8)
df['pagerank_ratio'] = np.minimum(df['u_pagerank'], df['v_pagerank']) / (np.maximum(df['u_pagerank'], df['v_pagerank']) + 1e-8)

print("✅ Centralidades diversificadas calculadas com sucesso!")

- PageRank centrality...
- Katz centrality...
- Load centrality (amostra)...
- Load centrality (amostra)...
Aplicando centralidades aos pares...
Calculando features derivadas...
✅ Centralidades diversificadas calculadas com sucesso!
Aplicando centralidades aos pares...
Calculando features derivadas...
✅ Centralidades diversificadas calculadas com sucesso!


In [44]:
# Detecção de comunidades usando algoritmo Louvain
print("Detectando comunidades com algoritmo Louvain...")

# Aplicar Louvain
partition = community_louvain.best_partition(G, random_state=42)

# Calcular modularidade
modularity = community_louvain.modularity(partition, G)
print(f"Modularidade da rede: {modularity:.4f}")

# Estatísticas das comunidades
communities = {}
for node, comm_id in partition.items():
    if comm_id not in communities:
        communities[comm_id] = []
    communities[comm_id].append(node)

print(f"Número de comunidades detectadas: {len(communities)}")
print(f"Tamanho médio das comunidades: {np.mean([len(comm) for comm in communities.values()]):.2f}")
print(f"Tamanho da maior comunidade: {max([len(comm) for comm in communities.values()])}")

# Adicionar informação de comunidade aos nós
nx.set_node_attributes(G, partition, 'community')

Detectando comunidades com algoritmo Louvain...
Modularidade da rede: 0.4204
Número de comunidades detectadas: 805
Tamanho médio das comunidades: 5.68
Tamanho da maior comunidade: 1156
Modularidade da rede: 0.4204
Número de comunidades detectadas: 805
Tamanho médio das comunidades: 5.68
Tamanho da maior comunidade: 1156


In [45]:
#adamic adar

pairs = list(zip(df["u"], df["v"]))
chunks = np.array_split(pairs, 8)

def adar_chunk(chunk):
    return [round(p, 4) for _, _, p in nx.adamic_adar_index(G, chunk)]

def parallel_progress(func, chunks, n_jobs=8):
    results = Parallel(n_jobs=n_jobs, backend='loky')(
        delayed(func)(chunk) for chunk in tqdm(chunks, desc="Calculando Adamic-Adar", ncols=80)
    )
    return results

results = parallel_progress(adar_chunk, chunks, n_jobs=8)
score_avg = [p for sublist in results for p in sublist]

df['aa_score'] = score_avg

Calculando Adamic-Adar: 100%|█████████████████████| 8/8 [00:00<00:00, 50.26it/s]



In [46]:
# Similaridades estruturais extras
print("Calculando similaridades estruturais extras...")

# 1. Jaccard Coefficient
def jaccard_chunk(chunk):
    return [round(p, 4) for _, _, p in nx.jaccard_coefficient(G, chunk)]

results_jaccard = parallel_progress(jaccard_chunk, chunks, n_jobs=8)
jaccard_scores = [p for sublist in results_jaccard for p in sublist]
df['jaccard_score'] = jaccard_scores

# 2. Preferential Attachment
def pref_attachment_chunk(chunk):
    return [round(p, 4) for _, _, p in nx.preferential_attachment(G, chunk)]

results_pref = parallel_progress(pref_attachment_chunk, chunks, n_jobs=8)
pref_scores = [p for sublist in results_pref for p in sublist]
df['pref_attachment'] = pref_scores

# 3. Clustering Coefficients dos nós
clustering = nx.clustering(G)
df['u_clustering'] = df['u'].map(clustering)
df['v_clustering'] = df['v'].map(clustering)

# Features derivadas do clustering
df['clustering_diff'] = np.abs(df['u_clustering'] - df['v_clustering'])
df['clustering_prod'] = df['u_clustering'] * df['v_clustering']
df['clustering_mean'] = (df['u_clustering'] + df['v_clustering']) / 2

print("✅ Similaridades estruturais extras calculadas!")

Calculando similaridades estruturais extras...


Calculando Adamic-Adar: 100%|███████████████████| 8/8 [00:00<00:00, 1373.27it/s]

Calculando Adamic-Adar: 100%|███████████████████| 8/8 [00:00<00:00, 4003.63it/s]



✅ Similaridades estruturais extras calculadas!


In [47]:
#inclusão de quantidade de estrelas, reviews e categorias

df['u_stars'] = df['u'].map(lambda node: G.nodes[node]['stars'])
df['v_stars'] = df['v'].map(lambda node: G.nodes[node]['stars'])

df['u_rev_count'] = df['u'].map(lambda node: G.nodes[node]['reviewCount'])
df['v_rev_count'] = df['v'].map(lambda node: G.nodes[node]['reviewCount'])

df['u_n_cat'] = df['u'].apply(lambda x: len([cat.strip() for cat in G.nodes[x].get('categories', '').split(',') if cat.strip()]) if x in G.nodes() else 0)
df['v_n_cat'] = df['v'].apply(lambda x: len([cat.strip() for cat in G.nodes[x].get('categories', '').split(',') if cat.strip()]) if x in G.nodes() else 0)

In [48]:
#inclusão de latitude e longitude

df['u_lat'] = df['u'].map(lambda node: G.nodes[node]['latitude'])
df['u_lon'] = df['u'].map(lambda node: G.nodes[node]['longitude'])
df['v_lat'] = df['v'].map(lambda node: G.nodes[node]['latitude'])
df['v_lon'] = df['v'].map(lambda node: G.nodes[node]['longitude'])

df['u_lat'] = df['u_lat']
df['u_lon'] = df['u_lon']
df['v_lat'] = df['v_lat']
df['v_lon'] = df['v_lon']

In [49]:
# Features de Comunidade
print("Extraindo features de comunidade...")

# 1. Comunidade de cada nó
df['u_community'] = df['u'].map(lambda node: G.nodes[node]['community'])
df['v_community'] = df['v'].map(lambda node: G.nodes[node]['community'])

# 2. Mesma comunidade (feature mais importante!)
df['same_community'] = (df['u_community'] == df['v_community']).astype(int)

# 3. Tamanho das comunidades
community_sizes = {comm_id: len(nodes) for comm_id, nodes in communities.items()}
df['u_comm_size'] = df['u_community'].map(community_sizes)
df['v_comm_size'] = df['v_community'].map(community_sizes)

# 4. Diferença entre tamanhos das comunidades
df['comm_size_diff'] = np.abs(df['u_comm_size'] - df['v_comm_size'])

# 5. Densidade interna das comunidades
community_densities = {}
for comm_id, nodes in communities.items():
    subgraph = G.subgraph(nodes)
    if len(nodes) > 1:
        density = nx.density(subgraph)
    else:
        density = 0.0
    community_densities[comm_id] = density

df['u_comm_density'] = df['u_community'].map(community_densities)
df['v_comm_density'] = df['v_community'].map(community_densities)

# 6. Diferença entre densidades das comunidades
df['comm_density_diff'] = np.abs(df['u_comm_density'] - df['v_comm_density'])

print(f"Pares na mesma comunidade: {df['same_community'].sum():,} ({df['same_community'].mean()*100:.2f}%)")
print("Features de comunidade extraídas com sucesso!")

Extraindo features de comunidade...
Pares na mesma comunidade: 173 (34.60%)
Features de comunidade extraídas com sucesso!


In [ ]:
# Otimização de tipos de dados - ATUALIZADO COM NOVAS CENTRALIDADES E SIMILARIDADES
float_cols = ['u_deg','v_deg','u_bet','v_bet','u_clo','v_clo','u_eig','v_eig','aa_score',
              'u_lat','u_lon','v_lat','v_lon','u_stars','v_stars',
              'u_comm_density','v_comm_density','comm_density_diff',
              # NOVAS centralidades
              'u_pagerank','v_pagerank','u_katz','v_katz','u_load','v_load',
              # Features derivadas
              'deg_diff','bet_diff','pagerank_diff','katz_diff','deg_ratio','pagerank_ratio',
              # NOVAS features de similaridade estrutural 
              'jaccard_score','pref_attachment','u_clustering','v_clustering','clustering_diff',
              'clustering_prod','clustering_mean']

int_cols = ['u_rev_count','v_rev_count','u_n_cat','v_n_cat',
            'u_community','v_community','same_community','u_comm_size','v_comm_size','comm_size_diff']

df[float_cols] = df[float_cols].astype('float32')
df[int_cols] = df[int_cols].astype('int16')

print(f"Shape do dataset: {df.shape}")


Shape do dataset: (500, 49)
   - Comunidades: 173 pares na mesma comunidade
   - Total de features: 47 features
   - Features de similaridade estrutural: 7 features adicionadas


In [52]:
"""
start_time = time.time()

node2vec = Node2Vec(
    G, 
    dimensions=64,         
    walk_length=20, 
    num_walks=100,        
    workers=8, 
    seed=42,
    quiet=False            
)

model = node2vec.fit(window=10, min_count=1, batch_words=4)

embeddings = {node: model.wv[node] for node in tqdm(G.nodes(), desc="Gerando dicionário de embeddings")}

emb_dim = len(next(iter(embeddings.values())))

X_emb = np.zeros((len(df), emb_dim), dtype=np.float32)

for i, (u, v) in enumerate(tqdm(zip(df['u'], df['v']), total=len(df), ncols=80, desc="Criando vetores diferenciais")):
    u_vec = embeddings.get(u)
    v_vec = embeddings.get(v)
    if u_vec is not None and v_vec is not None:
        X_emb[i] = np.abs(u_vec - v_vec)

emb_cols = [f'emb_{i}' for i in range(emb_dim)]
df_emb = pd.DataFrame(X_emb, columns=emb_cols)
df = pd.concat([df.reset_index(drop=True), df_emb], axis=1)
"""

'\nstart_time = time.time()\n\nnode2vec = Node2Vec(\n    G, \n    dimensions=64,         \n    walk_length=20, \n    num_walks=100,        \n    workers=8, \n    seed=42,\n    quiet=False            \n)\n\nmodel = node2vec.fit(window=10, min_count=1, batch_words=4)\n\nembeddings = {node: model.wv[node] for node in tqdm(G.nodes(), desc="Gerando dicionário de embeddings")}\n\nemb_dim = len(next(iter(embeddings.values())))\n\nX_emb = np.zeros((len(df), emb_dim), dtype=np.float32)\n\nfor i, (u, v) in enumerate(tqdm(zip(df[\'u\'], df[\'v\']), total=len(df), ncols=80, desc="Criando vetores diferenciais")):\n    u_vec = embeddings.get(u)\n    v_vec = embeddings.get(v)\n    if u_vec is not None and v_vec is not None:\n        X_emb[i] = np.abs(u_vec - v_vec)\n\nemb_cols = [f\'emb_{i}\' for i in range(emb_dim)]\ndf_emb = pd.DataFrame(X_emb, columns=emb_cols)\ndf = pd.concat([df.reset_index(drop=True), df_emb], axis=1)\n'

In [53]:
df.head()

,u,v,u_deg,v_deg,u_bet,v_bet,u_clo,v_clo,u_eig,v_eig,...,v_lon,u_community,v_community,same_community,u_comm_size,v_comm_size,comm_size_diff,u_comm_density,v_comm_density,comm_density_diff
0,mJ_ucQ2_3hfTsmCcKb-hgw,qXGKYRwCR9SLgLl0g_9o5g,0.006996,0.002186,0.000641,0.000387,0.286432,0.256788,4.375654e-02,1.316114e-02,...,-80.005287,2,0,0,640,893,253,0.008372,0.009320,0.000948
1,y19xFolCozaRA-gGmHwkQA,F6c3D1o9Z4Tl6cDorb3WgA,0.002624,0.001968,0.000193,0.000092,0.238642,0.250648,4.763492e-03,8.563046e-03,...,-80.021454,0,0,1,893,893,0,0.009320,0.009320,0.000000
2,R1GwW4C1gh2Nmue9K0WYVA,Ul6JwluSTm12PVDIqnNaTg,0.008745,0.017272,0.001618,0.005228,0.278267,0.290454,3.227378e-02,5.499225e-02,...,-79.984467,4,0,0,1156,893,263,0.007064,0.009320,0.002256
3,zzBa0pQjM1gov00bXjYYXg,3D6Uck9QSdxZKFstf5DGlg,0.001530,0.004373,0.000107,0.000699,0.241241,0.264060,5.744318e-03,1.651420e-02,...,-79.964058,2,2,1,640,640,0,0.008372,0.008372,0.000000
4,U2d-meX4sVq0kiqcrpHt1w,vuDL_d3GYAtbvX9EJQqVog,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.811329e-22,5.811329e-22,...,-79.950470,411,754,0,1,1,0,0.000000,0.000000,0.000000


In [54]:
df.to_csv('../data/edges_to_evaluate_Similaridades.csv', index=False)
print(f"\nDataset salvo com {df.shape[1]} features:")
print(f"   - {len(float_cols)} features numéricas")
print(f"   - {len(int_cols)} features categóricas/inteiras") 



Dataset salvo com 49 features:
   - 37 features numéricas
   - 10 features categóricas/inteiras
